# 使用Python构建RAG系统

## 1.解析文件

In [1]:
from typing import List
def split_into_chunks(doc_file:str) -> List[str]:
    with open(doc_file,'r', encoding='utf-8') as f:
        content = f.read()
    return [chunk for chunk in content.split("\n\n")]
chunks = split_into_chunks("test_doc.md")
for i,chunk in enumerate(chunks):
    print(f"[{i}] {chunk}\n")

[0] # 哆啦A梦与超级赛亚人：时空之战

[1] 在一个寻常的午后，大雄依旧坐在书桌前发呆，作业堆得像山，连第一页都没动。哆啦A梦在一旁翻着漫画，时不时叹口气，觉得这孩子还是一如既往的不靠谱。正当他们的生活照常进行时，一道强光突然从天而降，整个房间震动不已。光芒中走出一名金发少年，身披战甲、气势惊人，他就是来自未来的超级赛亚人——特兰克斯。他一出现便说出了惊人的话：未来的地球即将被黑暗势力摧毁，他来此是为了寻求哆啦A梦的帮助。

[2] 哆啦A梦与大雄听后大惊，但也从特兰克斯坚定的眼神中读出了不容拒绝的决心。特兰克斯解释说，未来的敌人并非普通反派，而是一个名叫“黑暗赛亚人”的存在，他由邪恶科学家复制了贝吉塔的基因并加以改造，实力超乎想象。这个敌人不仅拥有赛亚人战斗力，还能操纵扭曲的时间能量，几乎无人可敌。特兰克斯已经独自战斗多年，但每一次都以惨败告终。他说：“科技，是我那个时代唯一缺失的武器，而你们，正好拥有它。”

[3] 于是，哆啦A梦带着特兰克斯与大雄启动时光机，穿越到了那个即将崩溃的未来世界。眼前的景象令人震撼：城市沦为废墟，大地裂痕纵横，天空中浮动着压抑的黑雾。特兰克斯说，这正是黑暗赛亚人带来的结果，一切生命几乎都被抹杀，只剩他在苦苦支撑。大雄虽感到恐惧，但看到无辜的人类遭殃，内心逐渐燃起斗志。哆啦A梦则冷静地分析局势，决定使用他最强的三样秘密道具来对抗黑暗势力。

[4] 三件秘密道具分别是：可以临时赋予超级战力的“复制斗篷”，能暂停时间五秒的“时间停止手表”，以及可在一分钟中完成一年修行的“精神与时光屋便携版”。大雄被推进精神屋内，在其中接受密集的训练，虽然只有几分钟现实时间，他却经历了整整一年的苦修。刚开始他依旧软弱，想放弃、想逃跑，但当他想起静香、父母，还有哆啦A梦那坚定的眼神时，他终于咬牙坚持了下来。出来之后，他的身体与精神都焕然一新，眼神中多了一份成熟与自信。

[5] 最终战在黑暗赛亚人的空中要塞前爆发，特兰克斯率先出击，释放全力与敌人正面对决。哆啦A梦则用任意门和道具支援，从各个方向制造混乱，尽量压制敌人的时空能力。但黑暗赛亚人太过强大，仅凭特兰克斯一人根本无法压制，更别说击败。就在特兰克斯即将被击倒之际，大雄披上复制斗篷、冲破恐惧从高空跃下。他的拳头燃烧着金色光焰，目标直指敌人心脏。

[6] 时间停止装置在关键时刻启动，世界陷入静止

## 文档块向量化

In [2]:
from sentence_transformers import SentenceTransformer
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"  
embedding_model = SentenceTransformer("shibing624/text2vec-base-chinese")
def embed_chunk(chunk: str) -> List[float]:
    embedding = embedding_model.encode(chunk, normalize_embeddings=True)
    return embedding.tolist()


embedding = embed_chunk("测试内容")
print(len(embedding))
print(embedding)


768
[0.026805507019162178, 0.008382041938602924, 0.0003433833771850914, 0.007299028802663088, 0.0543331503868103, -0.05325588211417198, 0.001365506905131042, -0.0013182153925299644, -0.03671121224761009, 0.07188180834054947, -0.00727067282423377, -0.007053043693304062, 0.042532820254564285, -0.03675277158617973, -0.05475057661533356, -0.009598549455404282, 0.017105583101511, 0.05915362015366554, -0.03334999084472656, 0.062376584857702255, -0.004888508934527636, -0.034539539366960526, -0.07407601922750473, 0.04422195255756378, 0.010516923852264881, -0.0370778851211071, -0.027029825374484062, 0.038303639739751816, 0.02128247171640396, -0.011811508797109127, -0.0054087513126432896, 0.0026590346824377775, -0.02329857274889946, 0.05299094691872597, 0.005149427801370621, 0.029624182730913162, -0.030809687450528145, -0.017856178805232048, 0.04244604334235191, -0.007692297454923391, -0.010638121515512466, 0.032108623534440994, -0.06592466682195663, -0.01210090797394514, 0.006814637221395969, -

In [5]:
embeddings = [embed_chunk(chunk) for chunk in chunks]
print(len(embeddings))
print(embeddings[0])

10
[-0.01957523077726364, 0.007184448651969433, 0.023070000112056732, -0.012436461634933949, 0.039207518100738525, -0.05374177545309067, 0.028527164831757545, -0.02104203775525093, -0.0017696168506518006, 0.041362371295690536, -0.025198310613632202, -0.05593811348080635, 0.07257930189371109, 0.021626608446240425, -0.004362812731415033, -0.0002864626294467598, 0.060211535543203354, 0.02621517889201641, -0.04922759160399437, 0.009307670406997204, 0.013933545909821987, -0.005938063375651836, -0.03683415800333023, 0.02330167405307293, 0.010850666090846062, 0.00426430394873023, 0.003772017080336809, -0.02469753287732601, 0.0013592627365142107, 0.05580892413854599, 0.02183837816119194, 0.04607834666967392, -0.06695906072854996, 0.02910567820072174, 0.019366571679711342, -0.02105116844177246, 0.01536055188626051, -0.0030887252651154995, 0.01073166448622942, 0.02203545905649662, 0.034370146691799164, 0.04636268690228462, -0.05769667029380798, -0.05955016240477562, 0.0017393133603036404, 0.0557

### 存储到chromadb

In [6]:
import chromadb
chromadb_client = chromadb.EphemeralClient()
chromadb_collection = chromadb_client.get_or_create_collection("default")

def save_embedding(chunks:List[str],embeddings:List[List[float]]) -> None:
    for i, (chunk,embedding) in enumerate(zip(chunks,embeddings)):
        chromadb_collection.add(
            documents=[chunk],
            ids=[str(i)],
            embeddings=embedding,
        )
        
save_embedding(chunks,embeddings)


### 查询

In [7]:
def retrieve(query:str,top_k:int) -> List[str]:
    query_embedding = embed_chunk(query)
    query_result = chromadb_collection.query(query_embeddings=[query_embedding], n_results=top_k)
    return query_result['documents'][0]
  
query = "哆啦A梦使用的3个秘密道具分别是什么？"
retrieved_chunks = retrieve(query, 5)

for i, chunk in enumerate(retrieved_chunks):
    print(f"[{i}] {chunk}\n")  

[0] # 哆啦A梦与超级赛亚人：时空之战

[1] 三件秘密道具分别是：可以临时赋予超级战力的“复制斗篷”，能暂停时间五秒的“时间停止手表”，以及可在一分钟中完成一年修行的“精神与时光屋便携版”。大雄被推进精神屋内，在其中接受密集的训练，虽然只有几分钟现实时间，他却经历了整整一年的苦修。刚开始他依旧软弱，想放弃、想逃跑，但当他想起静香、父母，还有哆啦A梦那坚定的眼神时，他终于咬牙坚持了下来。出来之后，他的身体与精神都焕然一新，眼神中多了一份成熟与自信。

[2] 最终战在黑暗赛亚人的空中要塞前爆发，特兰克斯率先出击，释放全力与敌人正面对决。哆啦A梦则用任意门和道具支援，从各个方向制造混乱，尽量压制敌人的时空能力。但黑暗赛亚人太过强大，仅凭特兰克斯一人根本无法压制，更别说击败。就在特兰克斯即将被击倒之际，大雄披上复制斗篷、冲破恐惧从高空跃下。他的拳头燃烧着金色光焰，目标直指敌人心脏。

[3] 战后，未来世界开始恢复，植物重新生长，人类重建家园。特兰克斯告别时紧紧握住大雄的手，说：“你是我见过最特别的战士。”哆啦A梦也为大雄感到骄傲，说他终于真正成长了一次。三人站在山丘上，看着远方重新明亮的地平线，心中感受到从未有过的安宁。随后，哆啦A梦与大雄乘坐时光机返回了属于他们的那个年代，一切仿佛又恢复平静。

[4] 哆啦A梦与大雄听后大惊，但也从特兰克斯坚定的眼神中读出了不容拒绝的决心。特兰克斯解释说，未来的敌人并非普通反派，而是一个名叫“黑暗赛亚人”的存在，他由邪恶科学家复制了贝吉塔的基因并加以改造，实力超乎想象。这个敌人不仅拥有赛亚人战斗力，还能操纵扭曲的时间能量，几乎无人可敌。特兰克斯已经独自战斗多年，但每一次都以惨败告终。他说：“科技，是我那个时代唯一缺失的武器，而你们，正好拥有它。”



### 精排

In [9]:
from sentence_transformers import CrossEncoder 
cross_encoder = CrossEncoder("C:/Users/Lenovo/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3")
def rerank(query:str,retrieved_chunks:List[str], top_k:int) -> List[str]:
    pairs = [(query, chunk) for chunk in retrieved_chunks]
    scores = cross_encoder.predict(pairs)
    scored_chunks = list(zip(retrieved_chunks, scores))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)
    return [chunk for chunk, _ in scored_chunks][:top_k]
    
reranked_chunks = rerank(query, retrieved_chunks, 3)

for i, chunk in enumerate(reranked_chunks):
    print(f"[{i}] {chunk}\n")
    

[0] 三件秘密道具分别是：可以临时赋予超级战力的“复制斗篷”，能暂停时间五秒的“时间停止手表”，以及可在一分钟中完成一年修行的“精神与时光屋便携版”。大雄被推进精神屋内，在其中接受密集的训练，虽然只有几分钟现实时间，他却经历了整整一年的苦修。刚开始他依旧软弱，想放弃、想逃跑，但当他想起静香、父母，还有哆啦A梦那坚定的眼神时，他终于咬牙坚持了下来。出来之后，他的身体与精神都焕然一新，眼神中多了一份成熟与自信。

[1] 最终战在黑暗赛亚人的空中要塞前爆发，特兰克斯率先出击，释放全力与敌人正面对决。哆啦A梦则用任意门和道具支援，从各个方向制造混乱，尽量压制敌人的时空能力。但黑暗赛亚人太过强大，仅凭特兰克斯一人根本无法压制，更别说击败。就在特兰克斯即将被击倒之际，大雄披上复制斗篷、冲破恐惧从高空跃下。他的拳头燃烧着金色光焰，目标直指敌人心脏。

[2] # 哆啦A梦与超级赛亚人：时空之战



### 查询

In [20]:
import dotenv
import os
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("DASHSCOPE_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("BAILIAN_BASE_URL")
model = ChatOpenAI(model = "tongyi-xiaomi-analysis-flash")

# 生成回答函数（已修复）
def generate(query: str, chunks: List[str]) -> str:
    # 先把片段拼接好（修复核心！）
    context = "\n\n".join(chunks)
    
    prompt = f"""你是一位知识助手，请根据用户的问题和下列片段生成准确的回答。

用户问题: {query}

相关片段:
{context}

请基于上述内容作答，不要编造信息。"""

    print(f"{prompt}\n\n---\n")
    # 正确的 LangChain 调用方式
    response = model.invoke(prompt)
    return response.content  # 返回回答文本
answer = generate(query, reranked_chunks)
print(answer)

你是一位知识助手，请根据用户的问题和下列片段生成准确的回答。

用户问题: 哆啦A梦使用的3个秘密道具分别是什么？

相关片段:
三件秘密道具分别是：可以临时赋予超级战力的“复制斗篷”，能暂停时间五秒的“时间停止手表”，以及可在一分钟中完成一年修行的“精神与时光屋便携版”。大雄被推进精神屋内，在其中接受密集的训练，虽然只有几分钟现实时间，他却经历了整整一年的苦修。刚开始他依旧软弱，想放弃、想逃跑，但当他想起静香、父母，还有哆啦A梦那坚定的眼神时，他终于咬牙坚持了下来。出来之后，他的身体与精神都焕然一新，眼神中多了一份成熟与自信。

最终战在黑暗赛亚人的空中要塞前爆发，特兰克斯率先出击，释放全力与敌人正面对决。哆啦A梦则用任意门和道具支援，从各个方向制造混乱，尽量压制敌人的时空能力。但黑暗赛亚人太过强大，仅凭特兰克斯一人根本无法压制，更别说击败。就在特兰克斯即将被击倒之际，大雄披上复制斗篷、冲破恐惧从高空跃下。他的拳头燃烧着金色光焰，目标直指敌人心脏。

# 哆啦A梦与超级赛亚人：时空之战

请基于上述内容作答，不要编造信息。

---

根据提供的片段，哆啦A梦使用的三个秘密道具是：
1. 复制斗篷 - 可以临时赋予超级战力
2. 时间停止手表 - 能暂停时间五秒 
3. 精神与时光屋便携版 - 可在一分钟中完成一年修行

需要注意的是，虽然片段中提到了大雄在精神屋中接受训练并变得强大，但这是在"哆啦A梦与超级赛亚人：时空之战"的背景下，该片为跨作品内容，哆啦A梦本系列中哆啦A梦确实使用过这三件道具。
